# Coding Exercise: Supervised Fine-Tuning (SFT) and Direct Preference Optimization (DPO) of LLMs

**Course:** CS-552 Modern Natural Language Processing, EPFL

---

## 1. Overview

### Learning Objectives

In this exercise, you will learn:

1. **Supervised Fine-Tuning (SFT):** How to fine-tune a pre-trained language model on high-quality demonstration data to adapt it for a downstream task (e.g., dialogue, summarization).

2. **Direct Preference Optimization (DPO):** How to align a language model with human preferences *without* training an explicit reward model or using reinforcement learning. You will implement the DPO algorithm from the paper:
   > Rafailov et al., *"Direct Preference Optimization: Your Language Model is Secretly a Reward Model"*, NeurIPS 2023.

3. **Evaluation:** How to evaluate aligned models using both automatic metrics and reward-based analysis.

### Techniques & Skills
- Loading and preprocessing preference datasets
- Computing per-token and per-sequence log probabilities from a language model
- Implementing the SFT training loop with HuggingFace Transformers
- Implementing the DPO loss function component by component
- Understanding the role of the reference model in DPO
- Evaluating model quality with reward accuracy and win-rate metrics

### Models & Datasets
- **Model:** `GPT-2` (124M parameters) -- small enough to train on a single Colab GPU
- **Dataset:** `Anthropic/hh-rlhf` -- the Anthropic Helpful-Harmless preference dataset, which contains dialogues with chosen/rejected response pairs

> **Note:** We use GPT-2 for pedagogical purposes so the exercise runs within Colab's resource limits. The techniques transfer directly to larger models (Pythia, LLaMA, etc.).

## 1.1 Environment Setup

In [ ]:
# Install required packages
!pip install -q transformers datasets accelerate tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
import datasets as hf_datasets
from collections import defaultdict
import numpy as np
import random
import copy
import tqdm
from typing import Dict, List, Tuple, Optional, Union
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---

## 2. Theory and Mathematical Foundations

### 2.1 The RLHF Pipeline (Background)

The standard RLHF pipeline has three stages:

1. **Supervised Fine-Tuning (SFT):** Fine-tune a pre-trained LM on high-quality demonstrations to obtain $\pi_{\text{SFT}}$.

2. **Reward Modeling:** Collect preference data $(x, y_w, y_l)$ where $y_w \succ y_l \mid x$ (i.e., $y_w$ is preferred over $y_l$ for prompt $x$). Train a reward model $r_\phi(x, y)$ using the Bradley-Terry preference model:
$$p^*(y_1 \succ y_2 \mid x) = \frac{\exp(r^*(x, y_1))}{\exp(r^*(x, y_1)) + \exp(r^*(x, y_2))} = \sigma(r^*(x, y_1) - r^*(x, y_2))$$

   The reward model is trained by minimizing the negative log-likelihood:
$$\mathcal{L}_R(r_\phi, \mathcal{D}) = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \left[ \log \sigma(r_\phi(x, y_w) - r_\phi(x, y_l)) \right]$$

3. **RL Fine-Tuning:** Optimize the policy with a KL-constrained reward maximization objective:
$$\max_{\pi_\theta} \mathbb{E}_{x \sim \mathcal{D}, y \sim \pi_\theta(y|x)} \left[ r_\phi(x, y) \right] - \beta \, D_{\text{KL}}\left[ \pi_\theta(y|x) \| \pi_{\text{ref}}(y|x) \right]$$

### 2.2 DPO: From Reward to Policy (Key Derivation)

DPO eliminates the need for explicit reward modeling and RL by leveraging a closed-form relationship between the optimal policy and the reward function.

**Step 1: Optimal policy under KL-constrained reward maximization.**

The optimal solution to the KL-constrained objective (Eq. 3 above) is:
$$\pi_r(y \mid x) = \frac{1}{Z(x)} \pi_{\text{ref}}(y \mid x) \exp\left(\frac{1}{\beta} r(x, y)\right)$$
where $Z(x) = \sum_y \pi_{\text{ref}}(y \mid x) \exp\left(\frac{1}{\beta} r(x, y)\right)$ is the partition function.

**Step 2: Reparameterize the reward in terms of the policy.**

Rearranging the equation above, we can express the reward as:
$$r(x, y) = \beta \log \frac{\pi_r(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x)$$

**Step 3: Substitute into the Bradley-Terry model.**

Since the BT model depends only on *differences* of rewards, the partition function $Z(x)$ cancels:
$$p^*(y_1 \succ y_2 \mid x) = \sigma\left( \beta \log \frac{\pi^*(y_1 \mid x)}{\pi_{\text{ref}}(y_1 \mid x)} - \beta \log \frac{\pi^*(y_2 \mid x)}{\pi_{\text{ref}}(y_2 \mid x)} \right)$$

**Step 4: The DPO loss.**

We can now write a maximum likelihood objective directly over the policy $\pi_\theta$:

$$\boxed{\mathcal{L}_{\text{DPO}}(\pi_\theta; \pi_{\text{ref}}) = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)} \right) \right]}$$

### 2.3 Intuition Behind the DPO Gradient

The gradient of the DPO loss is:
$$\nabla_\theta \mathcal{L}_{\text{DPO}} = -\beta \, \mathbb{E} \left[ \underbrace{\sigma(\hat{r}_\theta(x, y_l) - \hat{r}_\theta(x, y_w))}_{\text{weight: higher when reward is wrong}} \left[ \underbrace{\nabla_\theta \log \pi_\theta(y_w \mid x)}_{\text{increase } \pi(y_w)} - \underbrace{\nabla_\theta \log \pi_\theta(y_l \mid x)}_{\text{decrease } \pi(y_l)} \right] \right]$$

where $\hat{r}_\theta(x, y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)}$ is the **implicit reward** defined by the policy.

Key insight: Examples are weighted by how *incorrectly* the implicit reward model currently ranks the pair. This prevents the model from degenerating -- a naive approach that simply increases $\pi(y_w)$ and decreases $\pi(y_l)$ without this weighting leads to model collapse.

---

### Exercise: Verify Your Understanding

**Q1.** Why does the partition function $Z(x)$ cancel in the DPO derivation?


**Q2.** What happens to the DPO loss as $\beta \to 0$? As $\beta \to \infty$?


**Q3.** In the DPO gradient, what does the weighting term $\sigma(\hat{r}_\theta(x, y_l) - \hat{r}_\theta(x, y_w))$ approach when the implicit reward model already correctly ranks $y_w$ much higher than $y_l$? What is the effect on learning?



---

## 3. Data Loading and Preprocessing

We will use the **Anthropic Helpful-Harmless (HH)** dataset, which contains multi-turn dialogues with a chosen (preferred) and rejected (dispreferred) response for each prompt.

In [ ]:
def extract_anthropic_prompt(prompt_and_response: str) -> str:
    """Extract the prompt from a full chosen/rejected string.
    The Anthropic HH format is:
        \n\nHuman: <query>\n\nAssistant: <response>
    We extract everything up to and including the last '\n\nAssistant:'.
    """
    search_term = '\n\nAssistant:'
    search_term_idx = prompt_and_response.rfind(search_term)
    assert search_term_idx != -1, f"Prompt does not contain '{search_term}'"
    return prompt_and_response[:search_term_idx + len(search_term)]


def get_hh_dataset(split: str, max_examples: int = None, silent: bool = False):
    """Load the Anthropic HH dataset and convert it to our format.

    Returns a dict mapping each prompt to:
      - 'responses': [chosen_response, rejected_response]
      - 'pairs': [(0, 1)]  (index 0 is preferred over index 1)
      - 'sft_target': chosen_response (used for SFT training)
    """
    print(f'Loading HH dataset ({split} split) from HuggingFace...')
    dataset = hf_datasets.load_dataset('Anthropic/hh-rlhf', split=split)
    print(f'Loaded {len(dataset)} examples.')

    if max_examples is not None:
        dataset = dataset.select(range(min(max_examples, len(dataset))))
        print(f'Using first {len(dataset)} examples.')

    data = defaultdict(lambda: defaultdict(list))
    for row in tqdm.tqdm(dataset, desc='Processing HH', disable=silent):
        prompt = extract_anthropic_prompt(row['chosen'])
        chosen_response = row['chosen'][len(prompt):]
        rejected_response = row['rejected'][len(prompt):]

        n_responses = len(data[prompt]['responses'])
        data[prompt]['pairs'].append((n_responses, n_responses + 1))
        data[prompt]['responses'].extend([chosen_response, rejected_response])
        data[prompt]['sft_target'] = chosen_response

    return data


# Load a small subset for this exercise
train_data = get_hh_dataset('train', max_examples=1500)
eval_data = get_hh_dataset('test', max_examples=300)

print(f'\nTrain prompts: {len(train_data)}')
print(f'Eval prompts: {len(eval_data)}')

In [ ]:
# Let's inspect some examples
sample_prompt = list(train_data.keys())[0]
sample_entry = train_data[sample_prompt]

print("=== Prompt ===")
print(sample_prompt)
print("\n=== Chosen Response ===")
print(sample_entry['responses'][0])
print("\n=== Rejected Response ===")
print(sample_entry['responses'][1])

### 3.1 Tokenization

We need to tokenize each example into:
- `input_ids`: the full sequence (prompt + response)
- `attention_mask`: 1 for real tokens, 0 for padding
- `labels`: same as `input_ids` for response tokens, `-100` for prompt tokens (so the loss is computed only on the response)

In [ ]:
def tokenize_batch_element(
    prompt: str,
    chosen: str,
    rejected: str,
    tokenizer,
    max_length: int = 512,
    max_prompt_length: int = 256,
) -> Dict:
    """Tokenize a single (prompt, chosen, rejected) triple.

    Creates input_ids, attention_mask, and labels for both chosen and rejected.
    Labels are set to -100 for prompt tokens (they are masked in the loss).
    """
    chosen_tokens = tokenizer(chosen, add_special_tokens=False)
    rejected_tokens = tokenizer(rejected, add_special_tokens=False)
    prompt_tokens = tokenizer(prompt, add_special_tokens=False)

    # Append EOS token to responses
    chosen_tokens['input_ids'].append(tokenizer.eos_token_id)
    chosen_tokens['attention_mask'].append(1)
    rejected_tokens['input_ids'].append(tokenizer.eos_token_id)
    rejected_tokens['attention_mask'].append(1)

    longer_response_length = max(len(chosen_tokens['input_ids']), len(rejected_tokens['input_ids']))

    # Truncate prompt if combined sequence is too long
    if len(prompt_tokens['input_ids']) + longer_response_length > max_length:
        prompt_tokens = {k: v[:max_prompt_length] for k, v in prompt_tokens.items()}

    # Truncate response if still too long
    if len(prompt_tokens['input_ids']) + longer_response_length > max_length:
        chosen_tokens = {k: v[:max_length - max_prompt_length] for k, v in chosen_tokens.items()}
        rejected_tokens = {k: v[:max_length - max_prompt_length] for k, v in rejected_tokens.items()}

    # Concatenate prompt + response
    chosen_sequence_tokens = {k: prompt_tokens[k] + chosen_tokens[k] for k in chosen_tokens}
    rejected_sequence_tokens = {k: prompt_tokens[k] + rejected_tokens[k] for k in rejected_tokens}

    # Create labels: -100 for prompt tokens (masked in loss)
    chosen_sequence_tokens['labels'] = chosen_sequence_tokens['input_ids'][:]
    chosen_sequence_tokens['labels'][:len(prompt_tokens['input_ids'])] = [-100] * len(prompt_tokens['input_ids'])
    rejected_sequence_tokens['labels'] = rejected_sequence_tokens['input_ids'][:]
    rejected_sequence_tokens['labels'][:len(prompt_tokens['input_ids'])] = [-100] * len(prompt_tokens['input_ids'])

    batch = {
        'prompt': prompt,
        'chosen': prompt + chosen,
        'rejected': prompt + rejected,
        'chosen_response_only': chosen,
        'rejected_response_only': rejected,
    }

    for k, toks in {'chosen': chosen_sequence_tokens, 'rejected': rejected_sequence_tokens, 'prompt': prompt_tokens}.items():
        for type_key, tokens in toks.items():
            if type_key == 'token_type_ids':
                continue
            batch[f'{k}_{type_key}'] = tokens

    return batch

In [ ]:
def get_collate_fn(tokenizer):
    """Returns a collate function that pads batch elements to equal length."""
    def collate_fn(batch):
        padded_batch = {}
        for k in batch[0].keys():
            if k.endswith('_input_ids') or k.endswith('_attention_mask') or k.endswith('_labels'):
                if 'prompt' in k:
                    # Left-pad prompts (reverse, pad, reverse)
                    to_pad = [torch.LongTensor(ex[k][::-1]) for ex in batch]
                else:
                    to_pad = [torch.LongTensor(ex[k]) for ex in batch]

                if k.endswith('_input_ids'):
                    padding_value = tokenizer.pad_token_id
                elif k.endswith('_labels'):
                    padding_value = -100
                elif k.endswith('_attention_mask'):
                    padding_value = 0
                else:
                    raise ValueError(f"Unexpected key '{k}'")

                padded_batch[k] = pad_sequence(to_pad, batch_first=True, padding_value=padding_value)
                if 'prompt' in k:
                    padded_batch[k] = padded_batch[k].flip(dims=[1])
            else:
                padded_batch[k] = [ex[k] for ex in batch]
        return padded_batch
    return collate_fn

In [ ]:
def prepare_dataset(
    data: dict,
    tokenizer,
    max_length: int = 512,
    max_prompt_length: int = 256,
    sft_mode: bool = False,
):
    """Convert raw preference data to a list of tokenized examples."""
    flat_data = []
    for prompt, entry in data.items():
        if sft_mode:
            # For SFT, we only use the preferred (sft_target) response
            element = tokenize_batch_element(
                prompt, entry['sft_target'], entry['sft_target'],
                tokenizer, max_length, max_prompt_length
            )
            element = {k: v for k, v in element.items() if 'rejected' not in k}
            flat_data.append(element)
        else:
            # For DPO, we use all preference pairs
            for (chosen_idx, rejected_idx) in entry['pairs']:
                element = tokenize_batch_element(
                    prompt, entry['responses'][chosen_idx], entry['responses'][rejected_idx],
                    tokenizer, max_length, max_prompt_length
                )
                flat_data.append(element)
    random.shuffle(flat_data)
    return flat_data


# Load tokenizer
MODEL_NAME = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 512
MAX_PROMPT_LENGTH = 256

# Prepare SFT dataset
sft_train_data = prepare_dataset(train_data, tokenizer, MAX_LENGTH, MAX_PROMPT_LENGTH, sft_mode=True)
sft_eval_data = prepare_dataset(eval_data, tokenizer, MAX_LENGTH, MAX_PROMPT_LENGTH, sft_mode=True)
print(f'SFT train examples: {len(sft_train_data)}')
print(f'SFT eval examples: {len(sft_eval_data)}')

# Prepare DPO dataset (we'll use this later)
dpo_train_data = prepare_dataset(train_data, tokenizer, MAX_LENGTH, MAX_PROMPT_LENGTH, sft_mode=False)
dpo_eval_data = prepare_dataset(eval_data, tokenizer, MAX_LENGTH, MAX_PROMPT_LENGTH, sft_mode=False)
print(f'DPO train examples: {len(dpo_train_data)}')
print(f'DPO eval examples: {len(dpo_eval_data)}')

---

## 4. Implementing SFT (Supervised Fine-Tuning)

SFT is conceptually simple: we fine-tune the language model to maximize the log-likelihood of the *chosen* response given the prompt. The loss is the standard causal language modeling (next-token prediction) loss, applied only to the response tokens.

$$\mathcal{L}_{\text{SFT}}(\pi_\theta) = -\mathbb{E}_{(x, y) \sim \mathcal{D}_{\text{SFT}}} \left[ \log \pi_\theta(y \mid x) \right]$$

### 4.1 Utility: Disable Dropout

Following the official DPO implementation, we disable dropout in the model during DPO training. This ensures consistent log-probability computation between the policy and reference model.

In [ ]:
def disable_dropout(model: nn.Module):
    """Disable dropout in a model (used in both SFT and DPO)."""
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.p = 0

### 4.2 Computing Log Probabilities

A fundamental building block: computing the log-probability of a response under a model.

For a sequence of tokens $(t_1, t_2, \ldots, t_T)$, the log-probability is:
$$\log \pi_\theta(y \mid x) = \sum_{i=1}^{T} \log \pi_\theta(t_i \mid t_1, \ldots, t_{i-1})$$

where we sum only over the *response* tokens (prompt tokens are masked with label `-100`).

In [ ]:
def get_batch_logps(
    logits: torch.FloatTensor,
    labels: torch.LongTensor,
    average_log_prob: bool = False,
) -> torch.FloatTensor:
    """Compute the log probabilities of the given labels under the given logits.

    Args:
        logits: Logits of the model (unnormalized). Shape: (batch_size, sequence_length, vocab_size)
        labels: Labels for which to compute the log probs. Tokens with value -100 are ignored.
                Shape: (batch_size, sequence_length)
        average_log_prob: If True, return average log prob per (non-masked) token.
                          Otherwise, return the sum.

    Returns:
        Tensor of shape (batch_size,) with the (average or sum) log probabilities.
    """
    assert logits.shape[:-1] == labels.shape, \
        f"Logits shape {logits.shape[:-1]} doesn't match labels shape {labels.shape}"

    ############################################################################
    # TODO: Implement the log probability computation.                          #
    #                                                                            #
    # Steps:                                                                     #
    # 1. Shift logits and labels: logits[:, :-1, :] predict labels[:, 1:]        #
    #    (standard causal LM setup: each position predicts the next token).      #
    # 2. Create a loss mask from labels: mask out positions where label == -100.  #
    # 3. Replace -100 labels with 0 (dummy; we'll mask these out).               #
    # 4. Compute per-token log probs using log_softmax + gather.                 #
    # 5. Sum (or average) the masked log probabilities along the sequence dim.   #
    ############################################################################

    # YOUR CODE HERE
    raise NotImplementedError("Implement get_batch_logps")

    ############################################################################
    #                          END OF YOUR CODE                                 #
    ############################################################################

In [ ]:
# ==================== TEST: get_batch_logps ====================
def test_get_batch_logps():
    """Test the get_batch_logps function with known values."""
    torch.manual_seed(42)
    batch_size, seq_len, vocab_size = 2, 5, 10

    # Create simple logits and labels
    logits = torch.randn(batch_size, seq_len, vocab_size)
    labels = torch.tensor([
        [-100, -100, 3, 5, 7],   # prompt tokens masked, 3 response tokens
        [-100, 2, 4, 6, 8],      # prompt token masked, 4 response tokens
    ])

    # Test sum mode
    logps_sum = get_batch_logps(logits, labels, average_log_prob=False)
    assert logps_sum.shape == (batch_size,), f"Expected shape ({batch_size},), got {logps_sum.shape}"
    assert (logps_sum <= 0).all(), "Log probs should be <= 0"

    # Test average mode
    logps_avg = get_batch_logps(logits, labels, average_log_prob=True)
    assert logps_avg.shape == (batch_size,), f"Expected shape ({batch_size},), got {logps_avg.shape}"
    assert (logps_avg <= 0).all(), "Average log probs should be <= 0"

    # Average should be >= sum (since we divide by count >= 1)
    assert (logps_avg >= logps_sum).all(), "Average log probs should be >= sum log probs (both negative)"

    # Verify manually for the first example
    shifted_logits = logits[0, :-1]  # shape: (4, 10)
    shifted_labels = labels[0, 1:]   # [-100, 3, 5, 7]
    log_softmax = shifted_logits.log_softmax(-1)

    expected_sum = 0.0
    count = 0
    for i, lab in enumerate(shifted_labels):
        if lab != -100:
            expected_sum += log_softmax[i, lab].item()
            count += 1

    assert abs(logps_sum[0].item() - expected_sum) < 1e-5, \
        f"Sum log prob mismatch: got {logps_sum[0].item()}, expected {expected_sum}"
    assert abs(logps_avg[0].item() - expected_sum / count) < 1e-5, \
        f"Avg log prob mismatch: got {logps_avg[0].item()}, expected {expected_sum / count}"

    print("test_get_batch_logps PASSED")

test_get_batch_logps()

### 4.3 SFT Training Loop

In [ ]:
def train_sft(
    model: nn.Module,
    tokenizer,
    train_data: list,
    eval_data: list,
    batch_size: int = 4,
    lr: float = 5e-5,
    num_epochs: int = 1,
    eval_every: int = 200,
    max_grad_norm: float = 10.0,
):
    """Train a model with supervised fine-tuning on chosen responses."""
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    collate_fn = get_collate_fn(tokenizer)

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    eval_loader = DataLoader(eval_data, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    train_losses = []
    eval_losses = []
    step = 0

    for epoch in range(num_epochs):
        for batch in tqdm.tqdm(train_loader, desc=f'SFT Epoch {epoch+1}/{num_epochs}'):
            # Move to device
            input_ids = batch['chosen_input_ids'].to(device)
            attention_mask = batch['chosen_attention_mask'].to(device)
            labels = batch['chosen_labels'].to(device)

            ####################################################################
            # TODO: Implement the SFT forward pass and loss computation.        #
            #                                                                    #
            # Steps:                                                             #
            # 1. Forward pass: get logits from the model.                        #
            # 2. Compute log probs using get_batch_logps (sum, not average).     #
            # 3. The SFT loss is the negative of the log probs, averaged over    #
            #    the batch.                                                       #
            ####################################################################

            # YOUR CODE HERE
            raise NotImplementedError("Implement SFT forward pass")

            ####################################################################
            #                      END OF YOUR CODE                             #
            ####################################################################

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            optimizer.zero_grad()

            train_losses.append(loss.item())
            step += 1
        
            if step % eval_every == 0:
                model.eval()
                eval_loss = 0
                n_eval_batches = 0
                with torch.no_grad():
                    for eval_batch in eval_loader:
                        e_input_ids = eval_batch['chosen_input_ids'].to(device)
                        e_attention_mask = eval_batch['chosen_attention_mask'].to(device)
                        e_labels = eval_batch['chosen_labels'].to(device)

                        e_logits = model(e_input_ids, attention_mask=e_attention_mask).logits.float()
                        e_logps = get_batch_logps(e_logits, e_labels)
                        eval_loss += (-e_logps).mean().item()
                        n_eval_batches += 1

                avg_eval_loss = eval_loss / max(n_eval_batches, 1)
                eval_losses.append((step, avg_eval_loss))
                print(f'  Step {step}: eval_loss={avg_eval_loss:.4f}')
                model.train()
            

    return train_losses, eval_losses

In [ ]:
# Load model and run SFT
print("Loading pre-trained GPT-2...")
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)  
sft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

print(f"Model parameters: {sum(p.numel() for p in sft_model.parameters()) / 1e6:.1f}M")

# Train SFT
sft_train_losses, sft_eval_losses = train_sft(
    sft_model, tokenizer, sft_train_data, sft_eval_data,
    batch_size=4, lr=5e-5, num_epochs=2, eval_every=100
)

In [ ]:
# Plot SFT training curves
fig, axes = plt.subplots(1, 1, figsize=(9, 4))

axes.plot(sft_train_losses)
axes.set_xlabel('Step')
axes.set_ylabel('Train Loss')
axes.set_title('SFT Training Loss')

plt.tight_layout()
plt.show()

### 4.4 Generate Samples from the SFT Model

In [ ]:
def generate_samples(model, tokenizer, prompts: list, max_new_tokens: int = 128):
    """Generate responses for a list of prompts."""
    model.eval()
    results = []
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256).to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=True,
                temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id
            )
        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        results.append(response)
    return results


# Generate a few samples
sample_prompts = list(eval_data.keys())[:1]
sft_responses = generate_samples(sft_model, tokenizer, sample_prompts)
base_responses = generate_samples(base_model, tokenizer, sample_prompts)

for i, (prompt, base_response, sft_response) in enumerate(zip(sample_prompts, base_responses, sft_responses)):
    print(f"\n{'='*60}")
    print(f"Prompt {i+1}: {prompt}")
    print(f"Base Response: {base_response}")
    print(f"SFT Response: {sft_response}")

---

## 5. Implementing DPO (Direct Preference Optimization)

Now we implement DPO step by step. Recall the DPO loss:

$$\mathcal{L}_{\text{DPO}}(\pi_\theta; \pi_{\text{ref}}) = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)} \right) \right]$$

We need:
1. A **policy model** $\pi_\theta$ (initialized from SFT)
2. A frozen **reference model** $\pi_{\text{ref}}$ (a copy of the SFT model)
3. The **DPO loss function**
4. A **training loop** that handles both models

### 5.1 Component 1: Concatenated Forward Pass

For efficiency, we concatenate the chosen and rejected inputs into a single batch and do one forward pass. This avoids running the model twice per example.

In [ ]:
def pad_to_length(tensor: torch.Tensor, length: int, pad_value: int, dim: int = -1) -> torch.Tensor:
    """Pad a tensor to the given length along the given dimension."""
    if tensor.size(dim) >= length:
        return tensor
    pad_size = list(tensor.shape)
    pad_size[dim] = length - tensor.size(dim)
    return torch.cat([tensor, pad_value * torch.ones(*pad_size, dtype=tensor.dtype, device=tensor.device)], dim=dim)


def concatenated_inputs(batch: Dict) -> Dict:
    """Concatenate the chosen and rejected inputs into a single tensor."""
    max_length = max(batch['chosen_input_ids'].shape[1], batch['rejected_input_ids'].shape[1])
    concatenated_batch = {}

    ############################################################################
    # TODO: Implement concatenation of chosen and rejected inputs.              #
    #                                                                            #
    # Steps:                                                                     #
    # 1. Find max_length = max of chosen and rejected sequence lengths.          #
    # 2. Pad chosen and rejected to max_length.                                  #
    #    - Pad input_ids with tokenizer.pad_token_id (use 0 here).              #
    #    - Pad attention_mask with 0.                                            #
    #    - Pad labels with -100.                                                 #
    # 3. Concatenate chosen (first) and rejected (second) along batch dim.       #
    ############################################################################

    max_length = max(batch['chosen_input_ids'].shape[1], batch['rejected_input_ids'].shape[1])
    concatenated_batch = {}

    # YOUR CODE HERE
    raise NotImplementedError("Implement concatenated_inputs")

    ############################################################################
    #                          END OF YOUR CODE                                 #
    ############################################################################

    return concatenated_batch

In [ ]:
# ==================== TEST: concatenated_inputs ====================
def test_concatenated_inputs():
    """Test the concatenated_inputs function."""
    batch_size = 3
    chosen_len = 10
    rejected_len = 8

    batch = {
        'chosen_input_ids': torch.randint(0, 100, (batch_size, chosen_len)),
        'chosen_attention_mask': torch.ones(batch_size, chosen_len, dtype=torch.long),
        'chosen_labels': torch.randint(0, 100, (batch_size, chosen_len)),
        'rejected_input_ids': torch.randint(0, 100, (batch_size, rejected_len)),
        'rejected_attention_mask': torch.ones(batch_size, rejected_len, dtype=torch.long),
        'rejected_labels': torch.randint(0, 100, (batch_size, rejected_len)),
    }

    result = concatenated_inputs(batch)

    # Check shapes
    assert result['concatenated_input_ids'].shape == (2 * batch_size, chosen_len), \
        f"Expected shape ({2 * batch_size}, {chosen_len}), got {result['concatenated_input_ids'].shape}"
    assert result['concatenated_attention_mask'].shape == (2 * batch_size, chosen_len)
    assert result['concatenated_labels'].shape == (2 * batch_size, chosen_len)

    # Check that first half is chosen, second half is rejected (padded)
    assert torch.equal(result['concatenated_input_ids'][:batch_size], batch['chosen_input_ids'])

    # Check padding
    assert (result['concatenated_attention_mask'][batch_size:, rejected_len:] == 0).all(), \
        "Rejected padding positions should have attention_mask=0"
    assert (result['concatenated_labels'][batch_size:, rejected_len:] == -100).all(), \
        "Rejected padding positions should have labels=-100"

    print("test_concatenated_inputs PASSED")

test_concatenated_inputs()

In [ ]:
def concatenated_forward(
    model: nn.Module,
    batch: Dict,
) -> Tuple[torch.FloatTensor, torch.FloatTensor]:
    """Run the model on concatenated chosen+rejected inputs."""
    concatenated_batch = concatenated_inputs(batch)

    all_logits = model(
        concatenated_batch['concatenated_input_ids'],
        attention_mask=concatenated_batch['concatenated_attention_mask'],
    ).logits.to(torch.float32)

    all_logps = get_batch_logps(all_logits, concatenated_batch['concatenated_labels'], average_log_prob=False)

    chosen_logps = all_logps[:batch['chosen_input_ids'].shape[0]]
    rejected_logps = all_logps[batch['chosen_input_ids'].shape[0]:]

    return chosen_logps, rejected_logps

### 5.2 Component 2: The DPO Loss Function

This is the core of the exercise. Implement the DPO loss:

$$\mathcal{L}_{\text{DPO}} = -\log \sigma\left( \beta \cdot \left[ \left(\log \pi_\theta(y_w|x) - \log \pi_{\text{ref}}(y_w|x)\right) - \left(\log \pi_\theta(y_l|x) - \log \pi_{\text{ref}}(y_l|x)\right) \right] \right)$$

In [ ]:
def dpo_loss(
    policy_chosen_logps: torch.FloatTensor,
    policy_rejected_logps: torch.FloatTensor,
    reference_chosen_logps: torch.FloatTensor,
    reference_rejected_logps: torch.FloatTensor,
    beta: float,
) -> Tuple[torch.FloatTensor, torch.FloatTensor, torch.FloatTensor]:
    """Compute the DPO loss for a batch of policy and reference model log probabilities.

    Args:
        policy_chosen_logps: Log probabilities of the policy model for chosen responses.
                             Shape: (batch_size,)
        policy_rejected_logps: Log probabilities of the policy model for rejected responses.
                               Shape: (batch_size,)
        reference_chosen_logps: Log probabilities of the reference model for chosen responses.
                                Shape: (batch_size,)
        reference_rejected_logps: Log probabilities of the reference model for rejected responses.
                                  Shape: (batch_size,)
        beta: Temperature parameter for DPO.

    Returns:
        losses: DPO loss for each example. Shape: (batch_size,)
        chosen_rewards: Implicit reward for chosen responses. Shape: (batch_size,)
        rejected_rewards: Implicit reward for rejected responses. Shape: (batch_size,)
    """
    ############################################################################
    # TODO: Implement the DPO loss function.                                    #
    #                                                                            #
    # Steps:                                                                     #
    # 1. Compute pi_logratios = policy_chosen_logps - policy_rejected_logps      #
    # 2. Compute ref_logratios = reference_chosen_logps - reference_rejected_logps#
    # 3. Compute logits = pi_logratios - ref_logratios                           #
    # 4. Compute losses = -F.logsigmoid(beta * logits)                           #
    # 5. Compute chosen_rewards and rejected_rewards (implicit rewards)           #
    ############################################################################

    # YOUR CODE HERE
    raise NotImplementedError("Implement dpo_loss")

    ############################################################################
    #                          END OF YOUR CODE                                 #
    ############################################################################

In [ ]:
# ==================== TEST: dpo_loss ====================
def test_dpo_loss():
    """Test the DPO loss function with known values."""
    torch.manual_seed(42)
    batch_size = 4
    beta = 0.1

    policy_chosen_logps = torch.tensor([-1.0, -2.0, -1.5, -0.5])
    policy_rejected_logps = torch.tensor([-2.0, -1.0, -1.5, -3.0])
    reference_chosen_logps = torch.tensor([-1.5, -1.5, -1.5, -1.5])
    reference_rejected_logps = torch.tensor([-1.5, -1.5, -1.5, -1.5])

    losses, chosen_rewards, rejected_rewards = dpo_loss(
        policy_chosen_logps, policy_rejected_logps,
        reference_chosen_logps, reference_rejected_logps,
        beta=beta
    )

    assert losses.shape == (batch_size,)
    assert chosen_rewards.shape == (batch_size,)
    assert rejected_rewards.shape == (batch_size,)
    assert (losses >= 0).all()

    pi_lr = policy_chosen_logps[2] - policy_rejected_logps[2]
    ref_lr = reference_chosen_logps[2] - reference_rejected_logps[2]
    expected_loss = -F.logsigmoid(torch.tensor(beta * (pi_lr - ref_lr)))
    assert abs(losses[2].item() - expected_loss.item()) < 1e-5

    assert losses[3] < losses[1]

    expected_chosen_reward = beta * (policy_chosen_logps[0] - reference_chosen_logps[0])
    assert abs(chosen_rewards[0].item() - expected_chosen_reward.item()) < 1e-5

    reward_accuracies = (chosen_rewards > rejected_rewards).float()
    assert reward_accuracies[0] == 1.0
    assert reward_accuracies[1] == 0.0

    print("test_dpo_loss PASSED")

test_dpo_loss()

### 5.3 Component 3: Computing DPO Batch Metrics

In [ ]:
def get_dpo_batch_metrics(
    policy_model: nn.Module,
    reference_model: nn.Module,
    batch: Dict,
    beta: float,
) -> Tuple[torch.FloatTensor, Dict]:
    """Compute DPO loss and metrics for a batch."""
    metrics = {}

    ############################################################################
    # TODO: Implement DPO batch metrics computation.                            #
    #                                                                            #
    # Steps:                                                                     #
    # 1. Get policy logps using concatenated_forward with policy_model.          #
    # 2. Get reference logps using concatenated_forward with reference_model     #
    #    (with torch.no_grad()).                                                 #
    # 3. Compute DPO loss using dpo_loss().                                      #
    # 4. Compute reward accuracy = fraction where chosen_reward > rejected_reward#
    # 5. Compute reward margin = mean(chosen_reward - rejected_reward).          #
    ############################################################################

    # YOUR CODE HERE
    raise NotImplementedError("Implement get_dpo_batch_metrics")

    ############################################################################
    #                          END OF YOUR CODE                                 #
    ############################################################################

In [ ]:
# ==================== TEST: get_dpo_batch_metrics ====================
def test_get_dpo_batch_metrics():
    """Test get_dpo_batch_metrics with a real model on a small batch."""
    test_model = AutoModelForCausalLM.from_pretrained('gpt2').to(device)
    ref_model = copy.deepcopy(test_model)
    disable_dropout(test_model)
    disable_dropout(ref_model)
    ref_model.eval()

    collate_fn = get_collate_fn(tokenizer)
    test_batch = collate_fn(dpo_eval_data[:4])
    test_batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in test_batch.items()}

    test_model.eval()
    with torch.no_grad():
        loss, metrics = get_dpo_batch_metrics(test_model, ref_model, test_batch, beta=0.1)

    expected_loss = torch.tensor(np.log(2))
    assert abs(loss.item() - expected_loss.item()) < 0.01, \
        f"When policy equals reference, loss should be ~log(2)={expected_loss.item():.4f}, got {loss.item():.4f}"

    assert 'reward_accuracy' in metrics
    assert 'reward_margin' in metrics
    assert abs(metrics['reward_margin']) < 0.01

    del test_model, ref_model
    torch.cuda.empty_cache()
    print("test_get_dpo_batch_metrics PASSED")

test_get_dpo_batch_metrics()

### 5.4 Component 4: DPO Training Loop

In [ ]:
def train_dpo(
    policy_model: nn.Module,
    reference_model: nn.Module,
    tokenizer,
    train_data: list,
    eval_data: list,
    beta: float = 0.1,
    batch_size: int = 4,
    lr: float = 5e-7,
    num_epochs: int = 1,
    eval_every: int = 100,
    max_grad_norm: float = 10.0,
):
    """Train a policy model using DPO."""
    reference_model.eval()
    policy_model.train()

    optimizer = torch.optim.RMSprop(policy_model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda step: min(1.0, (step + 1) / 151)
    )

    collate_fn = get_collate_fn(tokenizer)
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    eval_loader = DataLoader(eval_data, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    history = {
        'train_loss': [], 'train_reward_accuracy': [], 'train_reward_margin': [],
        'eval_loss': [], 'eval_reward_accuracy': [], 'eval_reward_margin': [],
        'eval_steps': [],
    }

    step = 0

    for epoch in range(num_epochs):
        for batch in tqdm.tqdm(train_loader, desc=f'DPO Epoch {epoch+1}/{num_epochs}'):
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

            loss, metrics = get_dpo_batch_metrics(policy_model, reference_model, batch, beta)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy_model.parameters(), max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            history['train_loss'].append(loss.item())
            history['train_reward_accuracy'].append(metrics['reward_accuracy'])
            history['train_reward_margin'].append(metrics['reward_margin'])
            step += 1

            if step % eval_every == 0:
                policy_model.eval()
                eval_metrics_agg = defaultdict(list)

                with torch.no_grad():
                    for eval_batch in eval_loader:
                        eval_batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                                      for k, v in eval_batch.items()}
                        _, eval_metrics = get_dpo_batch_metrics(
                            policy_model, reference_model, eval_batch, beta
                        )
                        for k, v in eval_metrics.items():
                            eval_metrics_agg[k].append(v)

                avg_eval_metrics = {k: np.mean(v) for k, v in eval_metrics_agg.items()}
                history['eval_loss'].append(avg_eval_metrics.get('loss', 0))
                history['eval_reward_accuracy'].append(avg_eval_metrics['reward_accuracy'])
                history['eval_reward_margin'].append(avg_eval_metrics['reward_margin'])
                history['eval_steps'].append(step)

                print(f"  Step {step}: "
                    #   f"train_loss={history['train_loss'][-1]:.4f}, "
                      f"eval_reward_acc={avg_eval_metrics['reward_accuracy']:.4f}, "
                      f"eval_reward_margin={avg_eval_metrics['reward_margin']:.4f}")
                policy_model.train()

    return history

### 5.5 Run DPO Training

In [ ]:
# Create the DPO policy (start from SFT model) and reference model (frozen SFT copy)
print("Setting up DPO training...")

dpo_policy = sft_model
dpo_reference = copy.deepcopy(sft_model)
del sft_model, base_model # unload models from gpu memory
disable_dropout(dpo_reference)
disable_dropout(dpo_policy)
dpo_reference.eval()
for param in dpo_reference.parameters():
    param.requires_grad = False

print(f"Policy parameters: {sum(p.numel() for p in dpo_policy.parameters()) / 1e6:.1f}M")
print(f"Reference parameters: {sum(p.numel() for p in dpo_reference.parameters()) / 1e6:.1f}M (frozen)")

BETA = 0.1
dpo_history = train_dpo(
    dpo_policy, dpo_reference, tokenizer,
    dpo_train_data, dpo_eval_data,
    beta=BETA, batch_size=4, lr=5e-7,
    num_epochs=2, eval_every=100,
)

In [ ]:
# Plot DPO training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(dpo_history['train_loss'], alpha=0.5, label='Train')
if dpo_history['eval_steps']:
    axes[0].plot(dpo_history['eval_steps'], dpo_history['eval_loss'], 'o-', label='Eval')
axes[0].axhline(y=np.log(2), color='r', linestyle='--', label='log(2) (random)')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('DPO Loss')
axes[0].set_title('DPO Loss')
axes[0].legend()

axes[1].plot(dpo_history['train_reward_accuracy'], alpha=0.5, label='Train')
if dpo_history['eval_steps']:
    axes[1].plot(dpo_history['eval_steps'], dpo_history['eval_reward_accuracy'], 'o-', label='Eval')
axes[1].axhline(y=0.5, color='r', linestyle='--', label='Random (0.5)')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Reward Accuracy')
axes[1].set_title('Implicit Reward Accuracy')
axes[1].legend()

axes[2].plot(dpo_history['train_reward_margin'], alpha=0.5, label='Train')
if dpo_history['eval_steps']:
    axes[2].plot(dpo_history['eval_steps'], dpo_history['eval_reward_margin'], 'o-', label='Eval')
axes[2].axhline(y=0, color='r', linestyle='--', label='Zero margin')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Reward Margin')
axes[2].set_title('Chosen - Rejected Reward Margin')
axes[2].legend()

plt.tight_layout()
plt.show()

---

## 6. Evaluation

### 6.1 Reward Accuracy on Held-Out Data

In [ ]:
def evaluate_reward_accuracy(
    policy_model: nn.Module,
    reference_model: nn.Module,
    eval_data: list,
    tokenizer,
    beta: float,
    batch_size: int = 4,
) -> Dict:
    """Evaluate the implicit reward accuracy on held-out preference data."""
    policy_model.eval()
    reference_model.eval()
    collate_fn = get_collate_fn(tokenizer)
    loader = DataLoader(eval_data, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    all_chosen_rewards = []
    all_rejected_rewards = []

    with torch.no_grad():
        for batch in tqdm.tqdm(loader, desc='Evaluating'):
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

            policy_chosen_logps, policy_rejected_logps = concatenated_forward(policy_model, batch)
            ref_chosen_logps, ref_rejected_logps = concatenated_forward(reference_model, batch)

            chosen_rewards = beta * (policy_chosen_logps - ref_chosen_logps)
            rejected_rewards = beta * (policy_rejected_logps - ref_rejected_logps)

            all_chosen_rewards.extend(chosen_rewards.cpu().numpy())
            all_rejected_rewards.extend(rejected_rewards.cpu().numpy())

    all_chosen_rewards = np.array(all_chosen_rewards)
    all_rejected_rewards = np.array(all_rejected_rewards)
    accuracy = (all_chosen_rewards > all_rejected_rewards).mean()
    avg_margin = (all_chosen_rewards - all_rejected_rewards).mean()

    return {
        'accuracy': accuracy,
        'avg_margin': avg_margin,
        'chosen_rewards': all_chosen_rewards,
        'rejected_rewards': all_rejected_rewards,
    }


eval_results = evaluate_reward_accuracy(
    dpo_policy, dpo_reference, dpo_eval_data, tokenizer, beta=BETA
)
print(f"\nEval Reward Accuracy: {eval_results['accuracy']:.4f}")
print(f"Eval Average Margin: {eval_results['avg_margin']:.4f}")

### 6.2 Qualitative Comparison: SFT vs. DPO

In [ ]:
dpo_responses = generate_samples(dpo_policy, tokenizer, sample_prompts)

for i, prompt in enumerate(sample_prompts):
    print(f"\n{'='*70}")
    print(f"PROMPT {i+1}:")
    print(prompt)
    print(f"\n--- SFT Response ---")
    print(sft_responses[i])
    print(f"\n--- DPO Response ---")
    print(dpo_responses[i])
    print(f"\n--- Ground Truth (Chosen) ---")
    print(eval_data[prompt]['sft_target'])

---

## 7. Results Analysis

### 7.1 Summary of Results

In [ ]:
print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(f"\nModel: {MODEL_NAME}")
print(f"Dataset: Anthropic HH-RLHF")
print(f"DPO beta: {BETA}")
print(f"\n--- SFT Training ---")
print(f"Final SFT train loss: {sft_train_losses[-1]:.4f}")
if sft_eval_losses:
    print(f"Final SFT eval loss: {sft_eval_losses[-1][1]:.4f}")
print(f"\n--- DPO Training ---")
print(f"Final DPO train loss: {dpo_history['train_loss'][-1]:.4f}")
print(f"Initial DPO loss (should be ~log(2)={np.log(2):.4f}): {dpo_history['train_loss'][0]:.4f}")
print(f"\n--- Evaluation ---")
print(f"Eval Reward Accuracy: {eval_results['accuracy']:.4f}")
print(f"Eval Reward Margin: {eval_results['avg_margin']:.4f}")

### 7.2 Discussion Questions

**Q1. Initial DPO Loss:** The DPO loss at the start of training should be approximately $\log 2 \approx 0.693$. Why?


**Q2. Reward Accuracy:** What does a reward accuracy significantly above 0.5 tell us?


**Q3. Role of Beta ($\beta$):**


**Q4. SFT vs. DPO:**


**Q5. Scaling Considerations:**



---

## 8. Summary

In this exercise, you have:

1. **Loaded and preprocessed** the Anthropic HH preference dataset.
2. **Implemented SFT** to fine-tune GPT-2 on chosen responses.
3. **Implemented the DPO loss** from scratch, understanding each component:
   - Computing log probabilities from model logits
   - The concatenated forward pass for efficiency
   - The core DPO loss: $-\log\sigma(\beta \cdot (\text{chosen\_logratios} - \text{rejected\_logratios}))$
   - Implicit reward computation and tracking
4. **Evaluated** the model using implicit reward accuracy and qualitative analysis.

### Key Takeaways

- **DPO eliminates the need for an explicit reward model and RL**, replacing them with a simple classification-like loss.
- **The reference model** acts as a regularizer, preventing the policy from deviating too far from the SFT baseline.
- **$\beta$** controls the trade-off: small $\beta$ allows more deviation (higher reward but risk of degeneration); large $\beta$ keeps the model close to the reference.
- **The implicit reward** $\hat{r}(x, y) = \beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}$ lets us monitor the model's preference learning without ever training an explicit reward function.
- **Reward accuracy** above 0.5 indicates the model is successfully learning from preferences.

### References

- Rafailov et al., ["Direct Preference Optimization: Your Language Model is Secretly a Reward Model"](https://arxiv.org/abs/2305.18290), NeurIPS 2023.
- [Official DPO implementation](https://github.com/eric-mitchell/direct-preference-optimization)
- Ziegler et al., ["Fine-Tuning Language Models from Human Preferences"](https://arxiv.org/abs/1909.08593), 2019.
- Ouyang et al., ["Training language models to follow instructions with human feedback"](https://arxiv.org/abs/2203.02155), NeurIPS 2022.